In [ ]:
# Install vLLM (run this first in a Colab cell)
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/53

In [ ]:
import os
os.environ["VLLM_USE_V1"] = "0"  # Use V0 engine (more notebook-friendly)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    max_num_seqs=128,
    block_size=16,
    gpu_memory_utilization=0.92,
    dtype="bfloat16",
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
)

conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain quantum computing in simple terms."},
]

outputs = llm.chat(conversation, sampling_params)
print(outputs[0].outputs[0].text)

INFO 05-08 02:17:53 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'block_size': 16, 'max_num_seqs': 128, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 05-08 02:17:53 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 05-08 02:17:54 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 05-08 02:17:54 [model.py:1680] Using max model len 4096
INFO 05-08 02:17:54 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 05-08 02:19:11 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations and process information. Unlike classical computing, which uses bits that can be either 0 or 1, quantum computing uses quantum bits, or qubits, which can be both 0 and 1 at the same time. This allows quantum computers to process a vast amount of information simultaneously, making them potentially much faster and more powerful than classical computers for certain types of calculations.


In [ ]:
# Batch inference - process many prompts at once
questions = [
    "What is the capital of France?",
    "Write a haiku about programming.",
    "Explain photosynthesis briefly.",
    "What are the benefits of exercise?",
    "How does a neural network learn?",
    "What causes rainbows?",
    "Give me a one-line joke about cats.",
    "What is the speed of light?",
]

# Build a list of conversations (one per question)
conversations = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": q},
    ]
    for q in questions
]

# Single call processes the entire batch in parallel
outputs = llm.chat(conversations, sampling_params)

# Print results
for i, output in enumerate(outputs):
    print(f"Q{i+1}: {questions[i]}")
    print(f"A: {output.outputs[0].text.strip()}")
    print("-" * 80)

Rendering conversations:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Q1: What is the capital of France?
A: The capital of France is Paris.
--------------------------------------------------------------------------------
Q2: Write a haiku about programming.
A: Syntax and logic,
Code that flows like a river,
Programs come alive.
--------------------------------------------------------------------------------
Q3: Explain photosynthesis briefly.
A: Photosynthesis is the process by which plants, algae, and some bacteria use sunlight to convert carbon dioxide and water into oxygen and glucose. This process occurs in the chloroplasts of plant cells and involves the following steps:

  1. Light absorption: Chlorophyll and other pigments in the chloroplasts absorb light energy, which is then used to split water molecules into hydrogen and oxygen.
  2. Water splitting: The hydrogen and oxygen from water molecules are used to produce glucose and oxygen through a series of chemical reactions.
  3. Glucose production: The glucose produced in the reaction is used by 